In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime

matplotlib.use('Agg')

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(os.path.dirname(BASE_DIR), 'data')

In [ ]:
def sizeof_fmt(num, suffix="B"):
    for unit in ("", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"):
        if abs(num) < 1024.0:
            return f"{num:3.1f}{unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f}Yi{suffix}"

def count_time(func):
    def wrapper(*args, **kwargs):
        start = datetime.now()
        result = func(*args, **kwargs)
        elapsed = datetime.now() - start
        print(f"Czas wykonania {func.__name__}: {elapsed}")
        return result
    return wrapper

### Zadanie 1

In [ ]:
df1 = pd.read_parquet(os.path.join(DATA_DIR, '0000.parquet'))
df2 = pd.read_parquet(os.path.join(DATA_DIR, '0001.parquet'))
df = pd.concat([df1, df2], ignore_index=True)
del df1, df2

In [ ]:
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
pd.options.display.float_format = '{:.2f}'.format
df.describe()

In [ ]:
mem_shallow = sum(df.memory_usage())
mem_deep = sum(df.memory_usage(deep=True))
print(f"Pamięć RAM (shallow): {sizeof_fmt(mem_shallow)}")
print(f"Pamięć RAM (deep):    {sizeof_fmt(mem_deep)}")

In [ ]:
for col in df.columns:
    print(f"{col:25s}: {sizeof_fmt(df[col].memory_usage(deep=True)):>10s}  ({df[col].dtype})")

### Zadanie 2

In [ ]:
df_orig = df.copy()
df_opt = pd.DataFrame()

for col in ['sid', 'sid_profile', 'profile_id']:
    min_val, max_val = df_orig[col].min(), df_orig[col].max()
    print(f"{col}: min={min_val}, max={max_val}")
    if min_val >= 0 and max_val <= np.iinfo(np.int32).max:
        df_opt[col] = df_orig[col].astype(np.int32)
    else:
        df_opt[col] = df_orig[col]

df_opt['post_id'] = df_orig['post_id'].astype('category')
df_opt['date'] = pd.to_datetime(df_orig['date'], errors='coerce')
df_opt['post_type'] = pd.to_numeric(df_orig['post_type'], downcast='integer')
df_opt['description'] = df_orig['description']
df_opt['likes'] = pd.to_numeric(df_orig['likes'], downcast='integer')
df_opt['comments'] = pd.to_numeric(df_orig['comments'], downcast='integer')
df_opt['username'] = df_orig['username'].astype('category')
df_opt['bio'] = df_orig['bio']
df_opt['following'] = pd.to_numeric(df_orig['following'], downcast='integer')
df_opt['followers'] = pd.to_numeric(df_orig['followers'], downcast='integer')
df_opt['num_posts'] = pd.to_numeric(df_orig['num_posts'], downcast='integer')
df_opt['is_business_account'] = df_orig['is_business_account']
df_opt['lang'] = df_orig['lang'].astype('category')
df_opt['category'] = df_orig['category'].astype('category')

In [ ]:
mem_orig = sum(df_orig.memory_usage(deep=True))
mem_opt = sum(df_opt.memory_usage(deep=True))

print(f"Pamięć oryginalna:      {sizeof_fmt(mem_orig)}")
print(f"Pamięć zoptymalizowana: {sizeof_fmt(mem_opt)}")
print(f"Redukcja: {(1 - mem_opt / mem_orig) * 100:.1f}%")

In [ ]:
for col in df_opt.columns:
    print(f"{col:25s}: {sizeof_fmt(df_opt[col].memory_usage(deep=True)):>10s}  ({df_opt[col].dtype})")

In [ ]:
labels = ['Oryginalna', 'Zoptymalizowana']
values = [mem_orig / (1024 ** 2), mem_opt / (1024 ** 2)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=['#e74c3c', '#2ecc71'], width=0.5, edgecolor='black')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{val:.1f} MiB', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Pamięć RAM (MiB)', fontsize=12)
ax.set_title('Porównanie zużycia pamięci RAM\n(oryginalne vs zoptymalizowane typy danych)', fontsize=13)
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'zadanie2_porownanie_pamieci.png'), dpi=150)
plt.show()

### Zadanie 3

In [ ]:
results = {}

start = datetime.now()
_ = df_orig.groupby('username').agg({'likes': 'mean'})
t1_orig = datetime.now() - start

start = datetime.now()
_ = df_opt.groupby('username').agg({'likes': 'mean'})
t1_opt = datetime.now() - start

print(f"Oryginalna:      {t1_orig}")
print(f"Zoptymalizowana: {t1_opt}")
results['Grupowanie +\nśrednia likes'] = (t1_orig.total_seconds(), t1_opt.total_seconds())

In [ ]:
start = datetime.now()
_ = df_orig[df_orig['likes'] > 100]
t2_orig = datetime.now() - start

start = datetime.now()
_ = df_opt[df_opt['likes'] > 100]
t2_opt = datetime.now() - start

print(f"Oryginalna:      {t2_orig}")
print(f"Zoptymalizowana: {t2_opt}")
results['Filtrowanie\nlikes > 100'] = (t2_orig.total_seconds(), t2_opt.total_seconds())

In [ ]:
start = datetime.now()
_ = df_orig.sort_values('followers', ascending=False)
t3_orig = datetime.now() - start

start = datetime.now()
_ = df_opt.sort_values('followers', ascending=False)
t3_opt = datetime.now() - start

print(f"Oryginalna:      {t3_orig}")
print(f"Zoptymalizowana: {t3_opt}")
results['Sortowanie\npo followers'] = (t3_orig.total_seconds(), t3_opt.total_seconds())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35
orig_times = [v[0] for v in results.values()]
opt_times = [v[1] for v in results.values()]

bars1 = ax.bar(x - width/2, orig_times, width, label='Oryginalna', color='#e74c3c')
bars2 = ax.bar(x + width/2, opt_times, width, label='Zoptymalizowana', color='#2ecc71')

ax.set_ylabel('Czas (sekundy)')
ax.set_title('Porównanie czasów operacji: oryginalne vs zoptymalizowane typy')
ax.set_xticks(x)
ax.set_xticklabels(results.keys())
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'zadanie3_porownanie_czasow.png'), dpi=150)
plt.show()

### Zadanie 4

In [ ]:
csv_path = os.path.join(BASE_DIR, 'instagram_data.csv')
df_orig.to_csv(csv_path, header=True, index=False)
csv_size = os.path.getsize(csv_path)
print(f"Rozmiar pliku CSV: {sizeof_fmt(csv_size)}")

In [ ]:
parquet_files = [os.path.join(DATA_DIR, '0000.parquet'), os.path.join(DATA_DIR, '0001.parquet')]
parquet_total = sum(os.path.getsize(f) for f in parquet_files)
print(f"Suma rozmiarów plików parquet: {sizeof_fmt(parquet_total)}")
print(f"CSV jest {csv_size / parquet_total:.1f}x większy niż parquet")

In [ ]:
del df_orig, df_opt, df
gc.collect()

### Zadanie 5

In [ ]:
from multiprocessing import cpu_count
n_cores = cpu_count()
print(f"Liczba rdzeni CPU: {n_cores}")

In [ ]:
csv_path = os.path.join(BASE_DIR, 'instagram_data.csv')

In [ ]:
@count_time
def read_csv_whole():
    return pd.read_csv(csv_path, header=0)

df_csv1 = read_csv_whole()
del df_csv1

In [ ]:
@count_time
def read_csv_chunked(chunksize):
    chunks = pd.read_csv(csv_path, header=0, chunksize=chunksize)
    return pd.concat(chunks, ignore_index=True)

for cs in [500_000, 1_000_000, 2_000_000]:
    print(f"chunksize={cs:,}")
    df_csv2 = read_csv_chunked(cs)
    del df_csv2

Multiprocessing nie działa w Jupyter Notebook na Windowsie — uruchom plik `zadanie5_6_multiprocessing.py`:

```
uv run python zadanie5_6_multiprocessing.py
```

### Zadanie 6

Zadanie 6 również wymaga multiprocessingu — wyniki w pliku `zadanie5_6_multiprocessing.py`:

```
uv run python zadanie5_6_multiprocessing.py
```